# Macmillan → tablas `macmillan_avance_operativo` y `macmillan_avance_historico`

Mismo camino que `Validacion Oxford_Nuevo`: los archivos se leen con **pandas**, se limpian y se dejan
en un **DataFrame**; solo después se suben a SQL Server con una `fecha_corte`.
Macmillan trae **una fila por estudiante** (cuatro promedios).

| Celda | Qué hace |
|---|---|
| 1 | Configuración: carpeta, fecha de corte, conexión y nombres de tablas |
| 2 | Funciones de limpieza |
| 3 | `crear_consolidado()`: lee los archivos y arma el DataFrame |
| 4-5 | Ejecutar y revisar el DataFrame (aquí no se toca la base) |
| 6 | Preparar las tablas en SQL Server (crea / renombra / agrega columnas si faltan) |
| 7 | Cargar: histórico por `fecha_corte` y refresco de la operativa |
| 8 | Comprobar contra la base |

**Cada vez que descargues archivos nuevos:** cambia `FECHA_CORTE` (celda 1) y corre todo.
Si la misma `fecha_corte` se carga dos veces, se reemplaza, no se duplica.

In [9]:
# 1. Configuración
import re
import time
from datetime import datetime
from pathlib import Path

import pandas as pd
import pyodbc

# Fecha de corte de esta carga (AAAA-MM-DD). Por defecto es hoy; cámbiala si cargas un corte atrasado.
FECHA_CORTE = datetime.now().strftime("%Y-%m-%d")

ORIGEN = Path(r"C:\Users\sebastian.mendez\Downloads\Macmillan")      # la carpeta entera (o un archivo suelto)
GUARDAR_EN = Path.cwd() / "avance_macmillan.csv"     # respaldo de lo que se cargó

TABLA_HISTORICO = "macmillan_avance_historico"
TABLA_OPERATIVA = "macmillan_avance_operativo"
TABLA_OPERATIVA_VIEJA = "macmillan_avcance_operativo"   # nombre con la errata; se renombra sola
CORTES_EN_OPERATIVA = 2        # la operativa guarda los últimos N cortes (como Oxford)

# Configuración de Conexión a Azure SQL (Nube)
DB_SERVER = 'bi-db-group.database.windows.net'
DB_NAME = 'rendimiento_academico'
DB_USER = 'bi_admin'
DB_PASS = '53/O~D&2m#o}'

CONN_STR = (
    f'DRIVER={{ODBC Driver 18 for SQL Server}};' 
    f'SERVER={DB_SERVER};' 
    f'DATABASE={DB_NAME};' 
    f'UID={DB_USER};' 
    f'PWD={DB_PASS};' 
    'Encrypt=yes;' 
    'TrustServerCertificate=no;' 
    'Connection Timeout=30;'
)

print("✅ Configuración cargada.")
print(f"📅 Fecha de corte: {FECHA_CORTE}")
print(f"📁 Origen: {ORIGEN}")
if ORIGEN.is_dir():
    EXTENSIONES = (".csv", ".xlsx", ".xls", ".xlsm")
    archivos = sorted(a for a in ORIGEN.iterdir() if a.suffix.lower() in EXTENSIONES)
elif ORIGEN.exists():
    archivos = [ORIGEN]
else:
    archivos = []
    print("❌ La ruta de origen NO existe. Revisa ORIGEN.")
print(f"📄 {len(archivos)} archivo(s) para procesar")
for archivo in archivos[:12]:
    print(f"   - {archivo.name}")
if len(archivos) > 12:
    print(f"   ... y {len(archivos) - 12} más")

✅ Configuración cargada.
📅 Fecha de corte: 2026-10-14
📁 Origen: C:\Users\sebastian.mendez\Downloads\Macmillan
📄 30 archivo(s) para procesar
   - Speak Your Mind_TCP_A1_1-10AM_SN_MJV_40.csv
   - Speak Your Mind_TCP_A1_1-130PM_TC_LMJ_01.csv
   - Speak Your Mind_TCP_A1_1-130PM_TC_LMJ_02.csv
   - Speak Your Mind_TCP_A1_1-130PM_TC_LMJ_03.csv
   - Speak Your Mind_TCP_A1_1-130PM_TC_LMJ_04.csv
   - Speak Your Mind_TCP_A1_1-130PM_TC_LMJ_05.csv
   - Speak Your Mind_TCP_A1_1-130PM_TC_LMJ_08.csv
   - Speak Your Mind_TCP_A1_1-130PM_TC_LMJ_09.csv
   - Speak Your Mind_TCP_A1_1-130PM_TC_LMV_18.csv
   - Speak Your Mind_TCP_A1_1-130PM_TC_LMV_19.csv
   - Speak Your Mind_TCP_A1_1-130PM_TC_LMV_20.csv
   - Speak Your Mind_TCP_A1_1-130PM_TC_LMV_21.csv
   ... y 18 más


In [10]:
# 2. Funciones de limpieza
COLUMNAS = {                       # como se llama en Macmillan -> como se llama en la tabla
    "Course Name": "course_name",
    "Class Name": "class_name",
    "Student Name": "student_name",
    "User Name": "user_name",
    "Attempt": "attempt",
    "Overall Course Average": "overall_course_average",
    "Coursework Average": "coursework_average",
    "Homework Average": "homework_average",
    "Assessment Average": "assessment_average",
}
PROMEDIOS = ["overall_course_average", "coursework_average",
             "homework_average", "assessment_average"]
TEXTOS = ["course_name", "class_name", "student_name", "user_name", "attempt"]
DERIVADAS = ["clase", "codigo", "grupo", "colegio"]

# Un estudiante, en una clase, con un intento: es la llave única de las tablas (por corte)
LLAVE_FILA = ["class_name", "user_name", "attempt"]
FECHA_EN_NOMBRE = re.compile(r"^(\d{4}-\d{2}-\d{2})_(\d{2})(\d{2})_")

#   A1.1-12PM04 JOSÉ CELESTINO MUTIS          TCP_A1_1-130PM_TC_LMJ_05
#   \__/ \__/\/ \__________________/               \__/ \___/ \____/\/
#  clase grupo cod      colegio                   clase grupo colegio cod
PATRON_CLASE = re.compile(r"""(?ix)
    (?P<clase>[a-z]{1,3}\s*\d+(?:[.\-_]\d+)?)     # A1.1, A1_1, B1-2
    \D*?                                         # texto suelto entre clase y grupo (MCE MEDIADO / PLUS-)
    (?P<grupo>\d{1,4}\s*[ap]\.?m\.?)              # 12PM, 8AM, 130PM (1:30 p.m.)
    (?P<resto>.*)$
""")


def codificacion_de(ruta):
    """Adivina la codificación por la firma (BOM). Macmillan exporta en UTF-8 sin firma."""
    firma = ruta.read_bytes()[:3]
    if firma.startswith((b"\xff\xfe", b"\xfe\xff")):
        return "utf-16"
    if firma.startswith(b"\xef\xbb\xbf"):
        return "utf-8-sig"
    return "utf-8"


def leer(ruta):
    """Devuelve el archivo con sus columnas, todo como texto (se convierte después)."""
    if ruta.suffix.lower() in (".xlsx", ".xls", ".xlsm"):
        return pd.read_excel(ruta, dtype=object)
    return pd.read_csv(ruta, dtype=object, encoding=codificacion_de(ruta),
                       sep=None, engine="python")


def texto(serie):
    """Serie de texto limpio y sin NaN."""
    return serie.astype("string").fillna("").str.strip()


def a_porcentaje(serie):
    """'53%' -> 53.0 | '0,55' -> 0.55 | 'No homework graded' -> NaN (queda NULL en la base)"""
    limpio = texto(serie).str.replace("%", "", regex=False).str.replace(",", ".", regex=False)
    return pd.to_numeric(limpio, errors="coerce").round(2)


def partes_de_clase(class_name):
    """Saca clase, grupo, código y colegio de 'A1.1-12PM04 JOSE CELESTINO MUTIS'."""
    limpio = " ".join(str(class_name).split())
    hallado = PATRON_CLASE.search(limpio)        # search: aguanta prefijos como 'TCP_'
    if not hallado:
        return {"clase": None, "codigo": None, "grupo": None, "colegio": None}

    resto = hallado["resto"].strip(" -_")
    pegado = re.match(r"^(\d+)", resto)          # lo normal: el código va pegado al grupo
    if pegado:
        codigo, colegio = pegado[1], resto[pegado.end():]
    else:                                        # variantes con el código al final
        final = re.search(r"(\d+)\s*$", resto)
        codigo = final[1] if final else None
        colegio = resto[:final.start()] if final else resto
    return {
        "clase": re.sub(r"[\s_\-]", ".", hallado["clase"].strip()).upper(),   # A1-1 -> A1.1
        "codigo": codigo,                        # texto: un 04 no se vuelve 4
        "grupo": re.sub(r"[\s.]", "", hallado["grupo"]).upper(),
        "colegio": " ".join(re.sub(r"[_\-]+", " ", colegio).split()).upper() or None,
    }


def descargado_en(ruta):
    """La fecha y hora que el bot pone al inicio del nombre; si no está, la del archivo."""
    hallado = FECHA_EN_NOMBRE.match(ruta.name)
    if hallado:
        return pd.Timestamp(f"{hallado[1]} {hallado[2]}:{hallado[3]}")
    return pd.Timestamp(ruta.stat().st_mtime, unit="s")


def a_tabla(crudo, origen=""):
    """Renombra las columnas, convierte los promedios y saca clase/grupo/código/colegio."""
    faltan = [c for c in COLUMNAS if c not in crudo.columns]
    if faltan:
        raise ValueError(f"A '{origen}' le faltan columnas: {faltan}")

    tabla = crudo[list(COLUMNAS)].rename(columns=COLUMNAS).copy()
    for columna in TEXTOS:
        tabla[columna] = texto(tabla[columna])
    for columna in PROMEDIOS:
        tabla[columna] = a_porcentaje(tabla[columna])

    partes = pd.DataFrame([partes_de_clase(c) for c in tabla["class_name"]], index=tabla.index)
    tabla = pd.concat([tabla, partes], axis=1)
    tabla = tabla[tabla["student_name"].ne("") & tabla["user_name"].ne("")]   # filas vacías
    tabla["origen"] = origen
    return tabla.reset_index(drop=True)


print("porcentajes ->", a_porcentaje(pd.Series(
    ["53%", "0%", "0,55", "No activities completed", "No homework graded"])).tolist())
for ejemplo in ("A1.1-12PM04 JOSÉ CELESTINO MUTIS", "A2-2 MCE MEDIADO / PLUS-1PM32- JULIO 2024",
                "TCP_A1_1-130PM_TC_LMJ_05", "Clase rara sin codigo"):
    print(f"  {ejemplo:<44} -> {partes_de_clase(ejemplo)}")

porcentajes -> [53.0, 0.0, 0.55, <NA>, <NA>]
  A1.1-12PM04 JOSÉ CELESTINO MUTIS             -> {'clase': 'A1.1', 'codigo': '04', 'grupo': '12PM', 'colegio': 'JOSÉ CELESTINO MUTIS'}
  A2-2 MCE MEDIADO / PLUS-1PM32- JULIO 2024    -> {'clase': 'A2.2', 'codigo': '32', 'grupo': '1PM', 'colegio': 'JULIO 2024'}
  TCP_A1_1-130PM_TC_LMJ_05                     -> {'clase': 'A1.1', 'codigo': '05', 'grupo': '130PM', 'colegio': 'TC LMJ'}
  Clase rara sin codigo                        -> {'clase': None, 'codigo': None, 'grupo': None, 'colegio': None}


In [11]:
# 3. Consolidar: lee todos los archivos y deja UN DataFrame.
#    La carpeta guarda el histórico de descargas: de cada estudiante se queda la MÁS RECIENTE.
def crear_consolidado(archivos):
    partes, errores = [], []
    print("⏳ Extrayendo datos de los archivos...")
    for archivo in archivos:
        try:
            inicio = time.time()
            tabla = a_tabla(leer(archivo), archivo.name)
            tabla["descargado_en"] = descargado_en(archivo)
            partes.append(tabla)
            print(f"   {archivo.name[:60]:<62} {len(tabla):>4} filas ({time.time() - inicio:.1f} s)")
        except Exception as error:
            errores.append(f"{archivo.name} -> {error}")

    if errores:
        print("\n⚠️ HUBO ERRORES AL LEER ESTOS ARCHIVOS (fueron saltados):")
        for error in errores:
            print(f" - {error}")
    if not partes:
        return pd.DataFrame(), 0

    todo = pd.concat(partes, ignore_index=True)
    final = (todo.sort_values("descargado_en")
                 .drop_duplicates(subset=LLAVE_FILA, keep="last")    # se queda la última descarga
                 .reset_index(drop=True))
    return final, len(todo) - len(final)

In [12]:
# 4. Ejecutar: el resultado es el DataFrame df_macmillan
df_macmillan, repetidas = crear_consolidado(archivos)

if df_macmillan.empty:
    print("\n❌ No se encontraron datos para consolidar.")
else:
    print(f"\n✅ Se consolidaron {len(df_macmillan):,} estudiantes "
          f"({repetidas:,} filas eran descargas repetidas de la misma clase).")
    display(df_macmillan.head(20))

⏳ Extrayendo datos de los archivos...
   Speak Your Mind_TCP_A1_1-10AM_SN_MJV_40.csv                      22 filas (0.0 s)
   Speak Your Mind_TCP_A1_1-130PM_TC_LMJ_01.csv                      5 filas (0.0 s)
   Speak Your Mind_TCP_A1_1-130PM_TC_LMJ_02.csv                     15 filas (0.0 s)
   Speak Your Mind_TCP_A1_1-130PM_TC_LMJ_03.csv                      8 filas (0.0 s)
   Speak Your Mind_TCP_A1_1-130PM_TC_LMJ_04.csv                      4 filas (0.0 s)
   Speak Your Mind_TCP_A1_1-130PM_TC_LMJ_05.csv                      4 filas (0.0 s)
   Speak Your Mind_TCP_A1_1-130PM_TC_LMJ_08.csv                      8 filas (0.0 s)
   Speak Your Mind_TCP_A1_1-130PM_TC_LMJ_09.csv                      4 filas (0.0 s)
   Speak Your Mind_TCP_A1_1-130PM_TC_LMV_18.csv                      6 filas (0.0 s)
   Speak Your Mind_TCP_A1_1-130PM_TC_LMV_19.csv                      6 filas (0.0 s)
   Speak Your Mind_TCP_A1_1-130PM_TC_LMV_20.csv                      5 filas (0.0 s)
   Speak Your Mind_TCP_A1_1

,course_name,class_name,student_name,user_name,attempt,overall_course_average,coursework_average,homework_average,assessment_average,clase,codigo,grupo,colegio,origen,descargado_en
0,Speak Your Mind Level 1,TCP_A1_1-130PM_TC_LMJ_05,Cristopher Andres Vargas Gomez,cristo.vargas312,First,<NA>,<NA>,<NA>,<NA>,A1.1,05,130PM,TC LMJ,Speak Your Mind_TCP_A1_1-130PM_TC_LMJ_05.csv,2026-10-13 19:45:11.712874889
1,Speak Your Mind Level 1,TCP_A1_1-130PM_TC_LMJ_05,Luis Miguel Matta Galvis,Luchomatta,First,76.0,76.0,<NA>,<NA>,A1.1,05,130PM,TC LMJ,Speak Your Mind_TCP_A1_1-130PM_TC_LMJ_05.csv,2026-10-13 19:45:11.712874889
2,Speak Your Mind Level 1,TCP_A1_1-130PM_TC_LMJ_05,Mariana Robayo Vargas,Machiss080,First,100.0,100.0,<NA>,<NA>,A1.1,05,130PM,TC LMJ,Speak Your Mind_TCP_A1_1-130PM_TC_LMJ_05.csv,2026-10-13 19:45:11.712874889
3,Speak Your Mind Level 1,TCP_A1_1-130PM_TC_LMJ_05,Diego Alejandro Barrero Bernal,diego.barrero,First,88.0,88.0,<NA>,<NA>,A1.1,05,130PM,TC LMJ,Speak Your Mind_TCP_A1_1-130PM_TC_LMJ_05.csv,2026-10-13 19:45:11.712874889
4,Speak Your Mind Level 1,TCP_A1_1-130PM_TC_LMV_18,Danna Briceño,dannabriceño2011,First,<NA>,<NA>,<NA>,<NA>,A1.1,18,130PM,TC LMV,Speak Your Mind_TCP_A1_1-130PM_TC_LMV_18.csv,2026-10-13 19:45:29.974873543
5,Speak Your Mind Level 1,TCP_A1_1-130PM_TC_LMV_18,GABRIELA FARFAN RODRIGUEZ,gabyfarfan123,First,<NA>,<NA>,<NA>,<NA>,A1.1,18,130PM,TC LMV,Speak Your Mind_TCP_A1_1-130PM_TC_LMV_18.csv,2026-10-13 19:45:29.974873543
6,Speak Your Mind Level 1,TCP_A1_1-130PM_TC_LMV_18,María Fernanda Calderon,mafecalderon2013,First,<NA>,<NA>,<NA>,<NA>,A1.1,18,130PM,TC LMV,Speak Your Mind_TCP_A1_1-130PM_TC_LMV_18.csv,2026-10-13 19:45:29.974873543
7,Speak Your Mind Level 1,TCP_A1_1-130PM_TC_LMV_18,Salome Briceño,salomebriceño2014,First,<NA>,<NA>,<NA>,<NA>,A1.1,18,130PM,TC LMV,Speak Your Mind_TCP_A1_1-130PM_TC_LMV_18.csv,2026-10-13 19:45:29.974873543
8,Speak Your Mind Level 1,TCP_A1_1-130PM_TC_LMV_18,Elizabeth Dayana Moreno,elizabethmoreno,First,<NA>,<NA>,<NA>,<NA>,A1.1,18,130PM,TC LMV,Speak Your Mind_TCP_A1_1-130PM_TC_LMV_18.csv,2026-10-13 19:45:29.974873543
9,Speak Your Mind Level 1,TCP_A1_1-130PM_TC_LMV_18,Simon Jacobo Siabatto,simonsiabatto,First,<NA>,<NA>,<NA>,<NA>,A1.1,18,130PM,TC LMV,Speak Your Mind_TCP_A1_1-130PM_TC_LMV_18.csv,2026-10-13 19:45:29.974873543


In [13]:
# 5. Revisión del DataFrame (todavía no se ha tocado la base)
if not df_macmillan.empty:
    print(df_macmillan.dtypes, "\n")
    print("estudiantes:", df_macmillan["user_name"].nunique(),
          "| clases:", df_macmillan["class_name"].nunique(),
          "| cursos:", df_macmillan["course_name"].nunique(),
          "| colegios:", df_macmillan["colegio"].nunique())
    print("\nintentos:\n", df_macmillan["attempt"].value_counts())
    print("\nvacíos por columna:\n", df_macmillan.isna().sum())
    print("\nrangos:")
    for columna in PROMEDIOS:
        print(f"  {columna:<24} min {df_macmillan[columna].min()}  max {df_macmillan[columna].max()}"
              f"  vacíos {df_macmillan[columna].isna().sum()}")

    sin_clase = df_macmillan["clase"].isna().sum()
    if sin_clase:
        print(f"\nOJO: {sin_clase} filas sin clase; revisa esos Class Name:")
        display(df_macmillan.loc[df_macmillan["clase"].isna(), ["class_name", "origen"]].drop_duplicates())

    fuera = df_macmillan[(df_macmillan[PROMEDIOS] > 100).any(axis=1) | (df_macmillan[PROMEDIOS] < 0).any(axis=1)]
    if len(fuera):
        print(f"\nOJO: {len(fuera)} filas con promedios fuera de 0-100:")
        display(fuera[["class_name", "user_name"] + PROMEDIOS])

    df_macmillan.to_csv(GUARDAR_EN, index=False, encoding="utf-8-sig")
    print(f"\nrespaldo guardado en {GUARDAR_EN}")

course_name                       string
class_name                        string
student_name                      string
user_name                         string
attempt                           string
overall_course_average           Float64
coursework_average               Float64
homework_average                 Float64
assessment_average               Float64
clase                                str
codigo                               str
grupo                                str
colegio                              str
origen                               str
descargado_en             datetime64[ns]
dtype: object 

estudiantes: 290 | clases: 30 | cursos: 1 | colegios: 5

intentos:
 attempt
First    301
Name: count, dtype: Int64

vacíos por columna:
 course_name                 0
class_name                  0
student_name                0
user_name                   0
attempt                     0
overall_course_average    158
coursework_average        190
homework_average      

## Carga a SQL Server

- **`macmillan_avance_historico`**: guarda todos los cortes. Si la `fecha_corte` ya existe, se borra y se vuelve a cargar.
- **`macmillan_avance_operativo`**: se vacía y se llena con los últimos cortes del histórico.

In [14]:
# 6. Preparar las tablas (se puede correr las veces que quieras)
COLUMNAS_TABLA = (TEXTOS + PROMEDIOS + DERIVADAS + ["fecha_corte", "created_at", "updated_at"])

DDL_TABLA = """CREATE TABLE dbo.{tabla} (
    id INT IDENTITY(1,1) CONSTRAINT PK_{tabla} PRIMARY KEY,
    course_name NVARCHAR(200) NOT NULL, class_name NVARCHAR(300) NOT NULL,
    student_name NVARCHAR(200) NOT NULL, user_name NVARCHAR(100) NOT NULL,
    attempt NVARCHAR(20) NOT NULL,
    overall_course_average DECIMAL(5,2) NULL, coursework_average DECIMAL(5,2) NULL,
    homework_average DECIMAL(5,2) NULL, assessment_average DECIMAL(5,2) NULL,
    fecha_corte DATE NOT NULL,
    CONSTRAINT UQ_{tabla} UNIQUE (fecha_corte, class_name, user_name, attempt))"""

COLUMNAS_FALTANTES = {              # lo que la tabla del primer diseño no traía
    "clase": "NVARCHAR(30) NULL", "codigo": "NVARCHAR(30) NULL",
    "grupo": "NVARCHAR(30) NULL", "colegio": "NVARCHAR(200) NULL",
    "created_at": "DATETIME2 NULL", "updated_at": "DATETIME2 NULL",
}


def preparar_tablas(cursor):
    # Renombrar la operativa si todavía tiene la errata (solo si el nombre bueno no existe)
    cursor.execute(f"""IF OBJECT_ID('dbo.{TABLA_OPERATIVA_VIEJA}') IS NOT NULL
                          AND OBJECT_ID('dbo.{TABLA_OPERATIVA}') IS NULL
                       EXEC sp_rename 'dbo.{TABLA_OPERATIVA_VIEJA}', '{TABLA_OPERATIVA}';""")
    for tabla in (TABLA_HISTORICO, TABLA_OPERATIVA):
        cursor.execute(f"IF OBJECT_ID('dbo.{tabla}') IS NULL EXEC('{DDL_TABLA.format(tabla=tabla)}');")
        for columna, tipo in COLUMNAS_FALTANTES.items():
            cursor.execute(f"IF COL_LENGTH('dbo.{tabla}', '{columna}') IS NULL "
                           f"ALTER TABLE dbo.{tabla} ADD {columna} {tipo};")


conn = pyodbc.connect(CONN_STR)
try:
    preparar_tablas(conn.cursor())
    conn.commit()
    for tabla in (TABLA_HISTORICO, TABLA_OPERATIVA):
        filas = conn.cursor().execute(f"SELECT COUNT(*) FROM dbo.{tabla}").fetchone()[0]
        print(f"✅ dbo.{tabla}: {filas:,} filas")
finally:
    conn.close()

✅ dbo.macmillan_avance_historico: 0 filas
✅ dbo.macmillan_avance_operativo: 0 filas


In [15]:
# 7. La carga (esta celda escribe en la base). Todo ocurre en UNA transacción:
#    si algo falla, no queda nada a medias.
def subir_a_tablas_finales(df, fecha_corte):
    if df.empty:
        print("❌ El DataFrame está vacío. No hay nada que subir.")
        return

    ahora = datetime.now()
    d = df.drop_duplicates(subset=LLAVE_FILA, keep="last").copy()
    d["fecha_corte"] = pd.Timestamp(fecha_corte).date()
    d["created_at"] = ahora
    d["updated_at"] = ahora
    d = d[COLUMNAS_TABLA]
    filas = d.astype(object).where(d.notna(), None).values.tolist()   # NaN -> NULL

    columnas = ", ".join(COLUMNAS_TABLA)
    marcas = ", ".join("?" * len(COLUMNAS_TABLA))

    conn = pyodbc.connect(CONN_STR, autocommit=False)
    try:
        cursor = conn.cursor()
        preparar_tablas(cursor)

        print(f"🧹 Borrando el corte {fecha_corte} del histórico (si ya estaba)...")
        borradas = cursor.execute(
            f"DELETE FROM dbo.{TABLA_HISTORICO} WHERE fecha_corte = ?", fecha_corte).rowcount

        print(f"🚀 Subiendo {len(filas):,} estudiantes al HISTÓRICO...")
        cursor.fast_executemany = True
        cursor.executemany(f"INSERT INTO dbo.{TABLA_HISTORICO} ({columnas}) VALUES ({marcas})", filas)
        cursor.fast_executemany = False

        print(f"🔄 Refrescando la OPERATIVA (últimos {CORTES_EN_OPERATIVA} cortes)...")
        cursor.execute("""SELECT COLUMN_NAME FROM INFORMATION_SCHEMA.COLUMNS
                          WHERE TABLE_NAME = ? AND COLUMNPROPERTY(
                                OBJECT_ID(TABLE_SCHEMA + '.' + TABLE_NAME), COLUMN_NAME, 'IsIdentity') = 0""",
                       TABLA_OPERATIVA)
        cols_op = [r.COLUMN_NAME for r in cursor.fetchall()]
        cursor.execute(f"""SELECT COLUMN_NAME FROM INFORMATION_SCHEMA.COLUMNS
                           WHERE TABLE_NAME = '{TABLA_HISTORICO}'""")
        cols_hist = {r.COLUMN_NAME for r in cursor.fetchall()}
        cols_op = [c for c in cols_op if c in cols_hist]       # las que existen en las dos

        cursor.execute(f"DELETE FROM dbo.{TABLA_OPERATIVA}")   # DELETE (no TRUNCATE): cabe en la transacción
        cursor.execute(f"""
            WITH UltimosCortes AS (
                SELECT TOP ({CORTES_EN_OPERATIVA}) fecha_corte
                FROM dbo.{TABLA_HISTORICO} GROUP BY fecha_corte ORDER BY fecha_corte DESC)
            INSERT INTO dbo.{TABLA_OPERATIVA} ({", ".join(cols_op)})
            SELECT {", ".join("h." + c for c in cols_op)}
            FROM dbo.{TABLA_HISTORICO} AS h
            JOIN UltimosCortes AS c ON c.fecha_corte = h.fecha_corte""")
        conn.commit()
        print(f"🎉 Listo. Se reemplazaron {borradas:,} filas del corte {fecha_corte} "
              f"por {len(filas):,} nuevas.")
    except Exception:
        conn.rollback()
        print("⛔ Falló la carga: se deshizo todo, la base quedó como estaba.")
        raise
    finally:
        conn.close()


subir_a_tablas_finales(df_macmillan, FECHA_CORTE)

🧹 Borrando el corte 2026-10-14 del histórico (si ya estaba)...
🚀 Subiendo 301 estudiantes al HISTÓRICO...
🔄 Refrescando la OPERATIVA (últimos 2 cortes)...
🎉 Listo. Se reemplazaron 0 filas del corte 2026-10-14 por 301 nuevas.


In [16]:
# 8. Comprobar contra la base
def consultar(sql, conn):
    cursor = conn.cursor().execute(sql)
    return pd.DataFrame.from_records([tuple(f) for f in cursor.fetchall()],
                                     columns=[c[0] for c in cursor.description])


conn = pyodbc.connect(CONN_STR)
try:
    cortes = consultar(f"""
        SELECT fecha_corte, COUNT(*) AS estudiantes, COUNT(DISTINCT class_name) AS clases
        FROM dbo.{TABLA_HISTORICO} GROUP BY fecha_corte ORDER BY fecha_corte DESC""", conn)
    print("Cortes en el histórico:")
    display(cortes)

    operativa = consultar(f"""
        SELECT fecha_corte, COUNT(*) AS estudiantes FROM dbo.{TABLA_OPERATIVA}
        GROUP BY fecha_corte ORDER BY fecha_corte DESC""", conn)
    print("Cortes en la operativa:")
    display(operativa)

    promedios = consultar(f"""
        SELECT colegio, clase, grupo, codigo, COUNT(*) AS estudiantes,
               ROUND(AVG(overall_course_average), 1) AS promedio_general,
               ROUND(AVG(coursework_average), 1)     AS promedio_curso,
               ROUND(AVG(homework_average), 1)       AS promedio_tareas
        FROM dbo.{TABLA_HISTORICO} WHERE fecha_corte = '{FECHA_CORTE}'
        GROUP BY colegio, clase, grupo, codigo ORDER BY colegio, clase, grupo, codigo""", conn)
    print(f"Promedios del corte {FECHA_CORTE}:")
    display(promedios)
finally:
    conn.close()

Cortes en el histórico:


,fecha_corte,estudiantes,clases
0,2026-10-14,301,30


Cortes en la operativa:


,fecha_corte,estudiantes
0,2026-10-14,301


Promedios del corte 2026-10-14:


,colegio,clase,grupo,codigo,estudiantes,promedio_general,promedio_curso,promedio_tareas
0,LF LMJ,A1.1,330PM,30,7,68.500000,68.500000,None
1,LF LMJ,A1.1,330PM,31,11,86.400000,86.400000,None
2,LF LMJ,A1.1,330PM,32,2,65.000000,65.000000,None
3,SN MJV,A1.1,10AM,40,22,87.900000,87.900000,None
4,TC LMJ,A1.1,130PM,01,5,44.000000,44.000000,None
5,TC LMJ,A1.1,130PM,02,15,0.000000,None,0.000000
6,TC LMJ,A1.1,130PM,03,8,43.700000,43.700000,None
7,TC LMJ,A1.1,130PM,04,4,86.300000,86.300000,None
8,TC LMJ,A1.1,130PM,05,4,88.000000,88.000000,None
9,TC LMJ,A1.1,130PM,08,8,56.800000,56.800000,56.700000
